In [1]:
%load_ext autoreload
%reload_ext autoreload

import logging
import os
import random
import time
from collections import defaultdict
from concurrent.futures import Executor, ProcessPoolExecutor, ThreadPoolExecutor
from dataclasses import asdict
from multiprocessing import get_context

import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd

# nsga2
from pymoo.core.population import Population
from pymoo.indicators.hv import HV
from pymoo.operators.survival.rank_and_crowding import RankAndCrowding
from shapely import wkt

import utils.route_utils as ru
from config.config import (
  CATCHMENT_RADIUS_M,
  FITNESS_WEIGHTS,
  INVALID_NETWORK_COST,
  MAX_BUDGET,
  MAX_DEGREE,
  N_NETWORKS,
  N_ROUTES,
  STOP_COST,
  STOP_MIN,
  STOP_SIZES,
  WALKING_DISTANCES,
  WD_TO_USE,
)
from genetic_algorithm.crossover import crossover
from genetic_algorithm.fitness import (
  fitness_score,
  make_run_tag,
)
from genetic_algorithm.mutate import mutate
from genetic_algorithm.schema import HyperParameters, Network, RunState
from utils.checkpoint_utils import (
  load_data,
  load_ga_checkpoint,
  save_final_networks,
  save_ga_checkpoint,
)
from utils.graph_utils import save_route_graph_to_plot
from utils.time_utils import ElapsedTimeFormatter, format_elapsed


In [2]:
# Logger object. Use debug, info, warning, error accordingly
logger = logging.getLogger("genetic_algorithm")
logger.setLevel(logging.DEBUG)

if not logger.handlers:
  handler = logging.StreamHandler()

  formatter = ElapsedTimeFormatter("[%(elapsed)s] %(levelname)s: %(message)s")

  handler.setFormatter(formatter)
  logger.addHandler(handler)

logger.propagate = False

In [3]:
# problem class for pymoo
class pymoo_prob:
      def __init__(self, constrained: bool) -> None:
          self.constrained = constrained

      def has_constraints(self) -> bool:
          return self.constrained

In [4]:
# for running processpooleval, comment out the ones i marked in the main code if you want to use thread
_EVAL_CTX: dict = {}

def set_eval_context(graph_of_stops: nx.MultiDiGraph, population_gdf: gpd.GeoDataFrame,
                     hyperparameters, city_name: str) -> None:
  hp = (hyperparameters if isinstance(hyperparameters, HyperParameters)
        else HyperParameters.from_dict(hyperparameters))
  _EVAL_CTX.update(hp=hp, graph_of_stops=graph_of_stops,
                   population_gdf=population_gdf, city_name=city_name)

def init_eval_worker(graph_of_stops: nx.MultiDiGraph, population_gdf: gpd.GeoDataFrame,
                     hyperparameters, city_name: str) -> None:
  """Pool initializer: ship the shared state once per worker instead of per task
  (stops graph 3.5 MB + population frame 2.7 MB pickled per task would dominate)."""
  set_eval_context(graph_of_stops, population_gdf, hyperparameters, city_name)
  random.seed(os.getpid() ^ time.time_ns())

def evaluate_network(network: Network):
  """Evaluate one network. Returns (fitness, violation) and never mutates the input."""
  ctx = _EVAL_CTX
  if not ctx:
    raise RuntimeError("evaluation context is empty: call set_eval_context() first")
  try:
    return fitness_score(network, ctx["hp"], ctx["graph_of_stops"],
                         ctx["population_gdf"], ctx["city_name"])
  except Exception:
    logger.exception("fitness evaluation failed; scoring the network as invalid")
    return np.array([INVALID_NETWORK_COST] * 3, dtype=float), 1.0


In [ ]:
# GA VERSION 3 by sir Anthony, Jericho, Daniel, and JP
def nsga_ii(
  initial_population: list[Network], graph_of_stops: nx.MultiDiGraph, hyperparameters: dict[str, any], city_name: str,
  population_gdf: gpd.GeoDataFrame, executor: Executor | None = None, run_tag = "", checkpoint = None, debug: bool = False
):
  # Sets logging to debug mode meaning logger.debug() works
  logger.setLevel(logging.DEBUG if debug else logging.INFO)
  hp: HyperParameters = (hyperparameters if isinstance(hyperparameters, HyperParameters) else HyperParameters.from_dict(hyperparameters))

  logger.info(f"nsga-ii Configuration | City: {city_name} | Pop Size: {len(initial_population)}")
  logger.info("Execution Mode : " + ("Sequential (single-threaded)" if not executor else f"Parallel ({type(executor).__name__})"))
  logger.info("Hyperparameters:")
  for k, v in asdict(hp).items():
    logger.info(f" └─ {k:<28}: {v}")

  if checkpoint is None:
    state = RunState(network_population=initial_population)
    logger.info("No checkpoint found. Starting fresh GA run...")
  else:
    state = RunState.from_checkpoint(checkpoint)
    logger.info("Resumed at generation %d (%.1fs accumulated).", state.generation_num, state.elapsed_before)

  started = time.monotonic()
  stage_times = defaultdict(float)

  def save():
    logger.info(f"Saving at generation {state.generation_num} ({elapsed():.2f}s)...")
    save_ga_checkpoint(f"_{run_tag}", state.to_checkpoint(elapsed()), city_name)

  def elapsed() -> float:
    return state.elapsed_before + (time.monotonic() - started)

  # def evaluate_network(network: Network):
  #   network.fitness_score, network.constraint_violation = fitness_score(
  #     network, hp, graph_of_stops, population_gdf, city_name,
  #   )

  set_eval_context(graph_of_stops, population_gdf, hp, city_name) # switch comments if you want threadpoolexec

  def run_evals(networks: list[Network]) -> None:
    if not networks:
      return
    results = ([evaluate_network(n) for n in networks] if executor is None
               else executor.map(evaluate_network, networks))
    for network, (fitness, violation) in zip(networks, list(results)):
      network.fitness_score, network.constraint_violation = fitness, violation

  HV_REF = np.array([1.6, 1.6, 1.6]) # hypervolume ref point of 1.6, should maybe be a hyperparam not sure
  obj_signs = np.array([1.0, 1.0, -1.0]) # user, capex, eff (negate)
  nds_crowd = RankAndCrowding(nds=None, crowding_func="cd")

  def pareto_front_hv(F_min: np.ndarray, ideal: np.ndarray, nadir: np.ndarray) -> float:
    span = np.where(nadir > ideal, nadir - ideal, 1.0) # nadir worst, ideal best
    val = HV(ref_point=HV_REF)((F_min - ideal) / span)
    return float(val) if val is not None else 0.0

  def rank_fitness(F, G, n_survive, random_state) -> tuple[list[int], Population]:
    F = np.asarray(F, dtype=float)
    G = np.asarray(G, dtype=float)

    if bool((G <= 0.0).any()):
      network_pop = Population.new("F", F * obj_signs, "G", G)
      survivors = nds_crowd.do(pymoo_prob(True), network_pop,
                               n_survive=n_survive, return_indices=True,
                               random_state=random_state)
      return np.asarray(survivors, dtype=int).tolist(), network_pop

    order = np.argsort(G[:, 0], kind="stable")             # <== ADDED: fallback path
    ranks = np.empty(len(F), dtype=float)
    ranks[order] = np.arange(len(F), dtype=float)
    network_pop = Population.new("F", F * obj_signs)
    network_pop.set("rank", ranks)
    network_pop.set("crowding", np.zeros(len(F), dtype=float))
    return order[:n_survive].astype(int).tolist(), network_pop 

  try:
    while state.generation_num < hp.max_generations:
      gen_start = time.perf_counter()
      generation_num = state.generation_num

      pop_size = len(state.network_population)
      survivors: list[int] = []
      offspring_networks: list[Network] = []

      # STAGE 0: Evaluate the fitness of each network in the population
      t0 = time.perf_counter()
      unevaluated = [n for n in state.network_population if n.fitness_score is None]
      run_evals(unevaluated) # switch comments for threadpoolexec
      # if executor and unevaluated:
      #   list(executor.map(evaluate_network, unevaluated))
      # else:
      #   [evaluate_network(n) for n in unevaluated]
      t_init_eval = time.perf_counter() - t0
      stage_times["initial_eval"] += t_init_eval

      t0 = time.perf_counter()

      # Set up pymoo population and rankandcrowding
      F = np.array([n.fitness_score for n in state.network_population], dtype=float)
      G = np.array([[n.constraint_violation] for n in state.network_population], dtype=float)
      has_feasible = bool((G <= 0.0).any())

      network_pop = rank_fitness(                           # <== CHANGED: was a duplicated if/else on (G <= 0.0).any()
        F, G, n_survive=pop_size, random_state=np.random.default_rng(0),
      )[1]

      fitness_scores = np.array([network.fitness_score for network in state.network_population])
      t_rank = time.perf_counter() - t0
      stage_times["rank"] += t_rank
      t0 = time.perf_counter()

      # STAGE 0.5: hypervolume computation
      # grab the pareto rank and crowding distance
      rank = np.array([np.inf if r is None else r for r in network_pop.get("rank")], float)
      crowd = np.array([-np.inf if c is None else c for c in network_pop.get("crowding")], float)

      front = fitness_scores[rank == 0] * obj_signs
      hv = state.hv_history[-1] if state.hv_history else float("nan")

      if state.hv_anchor is None and len(front):
        lo, hi = front.min(axis=0), front.max(axis=0)
        span = np.where(hi > lo, hi - lo, np.maximum(np.abs(lo) * 0.5, 1.0))
        state.hv_anchor = (lo, lo + span * 1.5) # should be a hyperparam i think

      if has_feasible and len(front):
        assert state.hv_anchor is not None
        hv = pareto_front_hv(front, *state.hv_anchor)
        state.hv_history.append(hv)
        if state.best_hv == 0.0 or hv > state.best_hv * 1.001:
          state.best_hv, state.plateau_counter = hv, 0
          state.best_hv_list.append(hv)
          state.best_population = list(state.network_population)
        else:
          state.plateau_counter += 1

      if state.plateau_counter >= hp.plateau_count_limit:
        logger.info(f"Plateau detected in generation {generation_num}. Stopping...")
        break

      # Getting the number of parents to be selected
      num_parents = int(len(state.network_population) * hp.selection_rate)
      num_parents = max(2, num_parents)
      if num_parents % 2 == 1:
        num_parents += 1
      t_hv = time.perf_counter() - t0
      stage_times["hv"] += t_hv

      #  STAGE 1: Binary tournament selection
      t0 = time.perf_counter()
      picks = []
      for _ in range(num_parents):
        a, b = np.random.randint(0, len(rank), 2)
        if (rank[b], -crowd[b]) < (rank[a], -crowd[a]):
          a = b
        picks.append(a)
      parent_networks = [state.network_population[i] for i in picks]

      # Pairing the parents
      parent_pairs = [(parent_networks[i], parent_networks[i + 1]) for i in range(0, len(parent_networks), 2)]
      t_sel = time.perf_counter() - t0
      stage_times["selection"] += t_sel

      # STAGE 2: Genetic Operators via. Crossover and Mutation
      t_crossover, t_mutation = 0.0, 0.0
      offspring_networks = []
      for parent1, parent2 in parent_pairs:
        tc0 = time.perf_counter()
        # Get 2 offspring from crossovers between the two parents
        child1, child2, is_different = crossover(parent1, parent2, graph_of_stops)
        t_crossover += time.perf_counter() - tc0

        tm0 = time.perf_counter()
        # Getting the number of mutations. If the offspring are entirely different from their parents, there should be atleast one mutation
        table = (hp.mutations_per_child if is_different else hp.mutations_per_child_identical)
        num_mutations = int(np.random.choice(len(table), p=table))

        # Apply mutations to the offspring based on mutation probability hyperparameter
        for child, parent in ((child1, parent1), (child2, parent2)):
          was_mutated = False
          for _ in range(num_mutations):
            if np.random.rand() < hp.mutation_probability and mutate(child, graph_of_stops, hp.walk_distance):
              was_mutated = True

          if is_different or was_mutated:
            child.fitness_score = None # Just make sure its none
            child.constraint_violation = None
          else:
            child.fitness_score = parent.fitness_score.copy()
            child.constraint_violation = parent.constraint_violation
        t_mutation += time.perf_counter() - tm0

        # Add the offspring to the list of offspring
        offspring_networks.extend((child1, child2))

      stage_times["crossover"] += t_crossover
      stage_times["mutation"] += t_mutation

      # STAGE 2: Post-op evaluation
      t0 = time.perf_counter()
      # Evaluate offspring networks for elitism
      unevaluated = [n for n in offspring_networks if n.fitness_score is None]
      run_evals(unevaluated) # switch comments for threadpoolexec
      # if executor and unevaluated:
      #   list(executor.map(evaluate_network, unevaluated))
      # else:
      #   [evaluate_network(n) for n in unevaluated]
      t_post_eval = time.perf_counter() - t0
      stage_times["post_eval"] += t_post_eval

      # STAGE 3: Elitism
      t0 = time.perf_counter()

      # Elitism: combining the current and offspring population into a single pool 2N, then selecting the best N individuals
      if offspring_networks:
        pop_size = len(state.network_population)

        # Combine curr population and offspring and select
        merged = state.network_population + offspring_networks
        F2 = np.array([n.fitness_score for n in merged], dtype=float)
        G2 = np.array([[n.constraint_violation] for n in merged], dtype=float)

        survivors, _ = rank_fitness(F2, G2, n_survive=pop_size, random_state=np.random.default_rng(generation_num))
        state.network_population = [merged[i] for i in survivors]
      t_elitism = time.perf_counter() - t0
      stage_times["elitism"] += t_elitism

      gen_total = time.perf_counter() - gen_start

      front_raw = fitness_scores[rank == 0]
      n_front, n_unique = len(front_raw), len(np.unique(fitness_scores.round(6), axis=0))
      best_user, best_capex, best_eff = front_raw[:, 0].min() if n_front else np.nan, front_raw[:, 1].min() if n_front else np.nan, front_raw[:, 2].max() if n_front else np.nan

      # Inside Stage 3 after computing `survivors`:
      offspring_survived = sum(1 for i in survivors if i >= pop_size)
      surv_rate = ((offspring_survived / len(offspring_networks) * 100.0) if offspring_networks else 0.0)

      logger.info(
        f"Gen {generation_num:04d} ({gen_total:5.2f}s) | "
        f"HV={hv:.4f} (Plat:{state.plateau_counter}/{hp.plateau_count_limit}) | "
        f"Feas={int((G <= 0.0).sum())}/{pop_size} | "
        f"F0={n_front}/{pop_size} (uniq={n_unique}) | "
        f"OffspringSurv={offspring_survived}/{len(offspring_networks)} ({surv_rate:.0f}%) | "
        f"Best[User={best_user:.3f}, CapEx={best_capex:.3f}, Eff={best_eff:.3f}]"
      )

      logger.debug(f"Gen {generation_num:04d} ({gen_total:5.2f}s) | " f"init_eval={t_init_eval:.3f}s, rank={t_rank:.4f}s, " f"hv={t_hv:.3f}s, sel={t_sel:.4f}s, " f"xover={t_crossover:.3f}s, mut={t_mutation:.3f}s, post_eval={t_post_eval:.3f}s, " f"elitism={t_elitism:.4f}s")

      # Increment the generation number
      state.generation_num += 1

      if state.generation_num % hp.checkpoint_every == 0:
        save()

  except KeyboardInterrupt:
    save()

  runtime = elapsed()
  logger.info(f"\n\nHighest HV of all generations: {state.best_hv}")
  logger.info(f"Total GA runtime: {format_elapsed(runtime)}")

  total_tracked = sum(stage_times.values()) or 1.0
  logger.info("Runtime Breakdown by Stage:")
  for stage, secs in sorted(stage_times.items(), key=lambda x: x[1], reverse=True):
    pct = (secs / total_tracked) * 100
    logger.info(f" └─ {stage:<16}: {secs:8.2f}s ({pct:5.1f}%)")

  return state.best_population or [], state.hv_history

In [ ]:
HYPERPARAMETERS = {
  "fitness_weights": FITNESS_WEIGHTS,
  "walk_distance": WALKING_DISTANCES[WD_TO_USE],

  # Selection
  "selection_rate": 0.5,

  # Mutation
  "mutation_probability": 0.8,
  "mutations_per_child": (0.10, 0.40, 0.35, 0.15),
  "mutations_per_child_identical": (0.00, 0.30, 0.45, 0.25),

  # Stop criterion
  "plateau_threshold": 1e-5,  
  "plateau_count_limit": 150,
  "max_generations": 9_999_999,

  # CapEx cost
  "stop_cost": STOP_COST,
  "stop_sizes": STOP_SIZES,
  "max_degree": MAX_DEGREE,
  "max_budget": 500_000_000,
  # Stop count manila: 9191
    
  # User cost
  "catchment_m": CATCHMENT_RADIUS_M,

  "checkpoint_every": 25,
}

In [ ]:
CITY_NAME = "Manila"  # Taguig, Mandaluyong, Manila, Makati
INITIAL_RS_DIR = f"../data/initial_route_sets/{CITY_NAME}"

irsg_run_tag = ru.make_run_tag(WALKING_DISTANCES[WD_TO_USE], STOP_MIN, N_ROUTES[CITY_NAME], N_NETWORKS)
file_path = os.path.join(INITIAL_RS_DIR, f"{CITY_NAME}_population{irsg_run_tag}.pkl")
initial_population = load_data(file_path)

file_path = os.path.join(INITIAL_RS_DIR, f"{CITY_NAME}_stops{irsg_run_tag}.pkl")
graph_of_stops = load_data(file_path)

filepath = f'../data/population_distribution/{CITY_NAME.lower()}-population-polygon.csv'
df = pd.read_csv(filepath)
df['geometry'] = df['geometry'].apply(wkt.loads)
population_gdf = gpd.GeoDataFrame(df, crs='epsg:32651')

ga_run_tag = make_run_tag(HYPERPARAMETERS["fitness_weights"])

In [8]:
def validate_network(network: Network, master_graph: nx.MultiDiGraph) -> None:
  for route_obj in network.routes:
    for conn in route_obj.route:
      u, v = conn[0], conn[-1]
      assert u in master_graph, (
        f"Stop {u} in {route_obj.route_id} missing from master_graph"
      )
      assert v in master_graph, (
        f"Stop {v} in {route_obj.route_id} missing from master_graph"
      )
      assert u in network.graph, f"Stop {u} missing from network.graph"
      assert v in network.graph, f"Stop {v} missing from network.graph"
      edge = network.graph.get_edge_data(u, v, key=route_obj.route_id)
      assert edge is not None and "distance" in edge, (
        f"Missing edge/distance ({u}, {v}, {route_obj.route_id})"
      )

for net in initial_population:
  validate_network(net, graph_of_stops)

print("IRS validated.")

IRS validated.


In [9]:
resume = False  # Set to true if u wanna use a checkpoint
checkpoint = load_ga_checkpoint(f"_{ga_run_tag}", CITY_NAME) if resume else None

N_WORKERS = min(50, os.cpu_count() or 1)
ctx = get_context("fork")

with ProcessPoolExecutor(
  max_workers=N_WORKERS,
  mp_context=ctx,
  initializer=init_eval_worker,
  initargs=(graph_of_stops, population_gdf, HYPERPARAMETERS, CITY_NAME),
) as executor:
  best_network, hv_history = nsga_ii(
    initial_population,
    graph_of_stops,
    HYPERPARAMETERS,
    CITY_NAME,
    population_gdf=population_gdf,
    executor=executor,
    run_tag=ga_run_tag,
    checkpoint=checkpoint,
    debug=True,
  )

# cpu_count = os.cpu_count() # threadpool
# assert cpu_count is not None
# with ThreadPoolExecutor(max_workers=cpu_count) as executor:
#   best_network, hv_history = nsga_ii(
#     initial_population,
#     graph_of_stops,
#     HYPERPARAMETERS,
#     CITY_NAME,
#     population_gdf=population_gdf,
#     executor=executor,
#     run_tag=ga_run_tag,
#     checkpoint=checkpoint,
#     debug=True,
#   )

  save_final_networks(ga_run_tag, best_network, CITY_NAME, hv_history)

[00:00:00.980] INFO: nsga-ii Configuration | City: Makati | Pop Size: 100
[00:00:00.980] INFO: Execution Mode : Parallel (ProcessPoolExecutor)
[00:00:00.981] INFO: Hyperparameters:
[00:00:00.981] INFO:  └─ fitness_weights             : {'w_cov': 0.2, 'w_int': 0.1, 'w_ext': 0.1, 'w_vul': 0.2, 'w_rob': 0.2, 'w_red': 0.2}
[00:00:00.981] INFO:  └─ walk_distance               : 750
[00:00:00.982] INFO:  └─ selection_rate              : 0.4
[00:00:00.982] INFO:  └─ mutation_probability        : 0.35
[00:00:00.982] INFO:  └─ mutations_per_child         : (0.25, 0.45, 0.2, 0.1)
[00:00:00.983] INFO:  └─ mutations_per_child_identical: (0.0, 0.4, 0.4, 0.2)
[00:00:00.983] INFO:  └─ catchment_m                 : 500.0
[00:00:00.983] INFO:  └─ stop_cost                   : 28000
[00:00:00.984] INFO:  └─ stop_sizes                  : (27.0, 40.5, 54.0)
[00:00:00.984] INFO:  └─ max_degree                  : (3, 5, 999999)
[00:00:00.984] INFO:  └─ max_budget                  : 500000000
[00:00:00.985] 

In [ ]:
save_route_graph_to_plot(graph_of_stops, best_network[0].routes, "best_network.html", f"../plots/Result Route Networks/{CITY_NAME}")

Saved best_network.html to: ../plots/Result Route Networks/Mandaluyong/best_network.html
